# ADDING FLAGS


In [10]:
import pandas as pd

# Load the dataset
file_path = "/content/ipl_bowl_features.csv"
df = pd.read_csv(file_path)

# Ensure the 'date' column is in datetime format
df["date"] = pd.to_datetime(df["date"], errors='coerce')

stats_columns = [
    "career_avg", "career_eco", "career_sr", "career_wpm","career_opm",
    "inns_avg", "inns_eco", "inns_sr", "inns_wpm","inns_opm",
    "venue_avg", "venue_eco", "venue_sr", "venue_wpm","venue_opm",
    "ssn_avg", "ssn_eco", "ssn_sr", "ssn_wpm","ssn_opm"
]

last_5_matches_columns = [
    "last5_avg", "last5_eco", "last5_sr", "last5_wpm","last5_opm"
]


# Identifying debuts
# Career debut: First match of the bowler based on earliest date
debut_dates = df.groupby("bowler_id")["date"].transform("min")
df["career_debut"] = (df["date"] == debut_dates).astype(int)

# Season debut: First match of the bowler in a season
df["season_debut"] = (df.groupby(["bowler_id", df["date"].dt.year])["date"].transform("min") == df["date"]).astype(int)

# Venue debut: First match of the bowler at a venue
df["venue_debut"] = (df.groupby(["bowler_id", "venue"])["date"].transform("min") == df["date"]).astype(int)

# Innings debut: First time the bowler is bowling in a specific innings (1 or 2) across all matches
df["innings_debut"] = (df.groupby(["bowler_id", "inns"])["date"].transform("min") == df["date"]).astype(int)


Cleaned data saved to: null_bowl_features.csv


## ADDING 0 TO NULL VALUES


In [11]:
# Fill null values with 0 if it's any kind of debut
stats_columns = [col for col in stats_columns if col in df.columns]
df.loc[df[["career_debut", "season_debut", "venue_debut", "innings_debut"]].any(axis=1), stats_columns] = df[stats_columns].fillna(0)

# Fill null values in last 5 matches columns with 0 only if it's a career debut

df.loc[df["career_debut"] == 1, last_5_matches_columns] = df[last_5_matches_columns].fillna(0)

output_path = "null2_bowl_features.csv"
df.to_csv(output_path, index=False)

print(f"Cleaned data saved to: {output_path}")

Cleaned data saved to: null2_bowl_features.csv
